# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
This notebook provides a step-by-step guide to exploring a dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library, referencing dataset schema entities by their `@id`. We'll load metadata, inspect record sets and fields, extract data, perform basic EDA, and visualize key relationships, all in a reproducible workflow.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Install required library
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Obtain top-level metadata
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")

## 2. Data Overview
Examine available record sets, their fields, and corresponding `@id`s as defined in the Croissant schema.

**Note**: In Croissant, each entity (record set, field, column) is referenced by its `@id`. We use these `@id` values to access and extract data robustly and reproducibly.

In [ ]:
# List all available record sets (their @id and name)
print('Record Sets:')
record_sets = []
for rs in dataset.record_sets:
    print(f"@id: {rs['@id']}  |  name: {rs.get('name', '(unnamed)')}")
    record_sets.append(rs['@id'])

if not record_sets:
    print("No record sets were found in the Croissant schema.")
else:
    # For the first record set, display available fields and columns
    first_rs_id = record_sets[0]
    print(f"\nFields and columns for record set '@id': {first_rs_id}")
    fields = dataset.fields(record_set=first_rs_id)
    for f in fields:
        print(f"  Field @id: {f['@id']}, label: {f.get('label', f.get('name','(unnamed)'))}, dataType: {f.get('dataType','n/a')}")
        if 'column' in f:
            if isinstance(f['column'], list):
                for col in f['column']:
                    if isinstance(col, dict):
                        print(f"    Column @id: {col.get('@id')}, name: {col.get('name')}")
                    else:
                        print(f"    Column @id: {col}")
            else:
                print(f"    Column @id: {f['column']}")

## 3. Data Extraction
Extract data from each record set into a Pandas DataFrame for further analysis, referencing record set and field `@id`s.

We'll dynamically create a DataFrame for each record set using its `@id`, so you can select and operate on specific record sets using their unique identifiers.

In [ ]:
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")

if record_sets:
    chosen_rs = record_sets[0]  # For demonstration, we use the first record set
    print(f"\nColumns in DataFrame for record set '@id': {chosen_rs}")
    print(dataframes[chosen_rs].columns.tolist())
    display(dataframes[chosen_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's do some basic analysis:
- Filter records on a numeric field (referenced by its `@id`)
- Normalize the numeric field
- Optionally group data by a categorical field

**Remember:** Always use the Croissant `@id` values to reference fields.

In [ ]:
# Pick a record set and its numeric & group (categorical) fields by @id
# You can change these @id values based on the previous overview
record_set_id = chosen_rs

# Attempt to find a numeric field based on DataFrame dtypes
df = dataframes[record_set_id]
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
    print(f"Using numeric field '@id': {numeric_field_id}")
else:
    print("No numeric field found for EDA.")
    numeric_field_id = None

# Try finding a group (categorical) field
group_field_id = None
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
        group_field_id = col
        break
if group_field_id:
    print(f"Will group by field '@id': {group_field_id}")

if numeric_field_id:
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records where {numeric_field_id} > {round(threshold,2)}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized '{numeric_field_id}' (mean 0, std 1) for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
We'll plot the distribution of the selected numeric field and, if a grouping exists, visualize group means.

You can modify the code below to select other fields using their Croissant `@id`s as explored above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and not df[numeric_field_id].isnull().all():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field with data for visualization.")

## 6. Conclusion
- In this notebook, we've loaded metadata and records from a Croissant-based dataset using the `mlcroissant` library.
- All data access and references were performed via each entity's `@id`.
- We demonstrated extracting DataFrames, filtering, normalizing, grouping, and visualizing numeric data using consistent, schema-driven referencing.

**Next steps:** You can extend this notebook to:
* Explore additional record sets (by changing the `@id`),
* Analyze categorical variables distributions,
* Merge across record sets (using matching `@id` relationships), or
* Perform further statistical or machine learning analyses if the data and use case allow.